<a href="https://colab.research.google.com/github/33MarGomez/Interactive-Tutorials/blob/main/selected_works_of_UnitaryHack_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Selected Works of Unitary Hack 2026

By: Marco Gomez

Release version: 06/29/26

Installs and Imports

In [ ]:
#quimb demo
!git clone https://github.com/jcmgray/quimb/
!pip install quimb
!git clone https://github.com/jcmgray/autoray
!pip install autoray

In [ ]:
#quimb only
import autoray as ar
import matplotlib as mpl
import tqdm

import quimb as qu
import quimb.tensor as qtn

mpl.style.use(qu.NEUTRAL_STYLE)

In [ ]:
#for all demos
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

#The Bounties

##quimb Bounty (\$150): Implement a PEPS quantum circuit simulator (\#358)

*Awarded to zkasuran for Pull # 371* [Direct](https://github.com/jcmgray/quimb/pull/371)

Quimb strictly focuses on tensor systems, making it useful in AI. Though most quantum packages have some sort of compilation module to specify the order of operations, quimb has quantum features without any need to construct a system. For that reason, with the correct knowledge of quantum:scientific-computation knowledge, it can be used generally to explore classes of problems to verify the fastest way to verify high-thoroughput work. It does explicitly support many-body calculations, but knowledge of operations greatly expedites using the program. Supported by a Caltech staff scientist, the university's geographic proximity to institutions like USC and UCLA (even arguably Berkeley) mean the project is likely to remain supported by peers in scientific computing spaces.

Implements a Class containing the basic ingredients of the quimb tutorial about PEPS (Projected Entangled Pair States) with the major difference being an absence of gen_gloops_sites(), compute_local_expectation_gloop_expand(), norm_gloop_expand(). Gloops are generalized loops where every node is connected to at least two bonds [according to docs](https://quimb.readthedocs.io/en/latest/autoapi/quimb/tensor/networking/index.html#quimb.tensor.networking.gen_gloops). This means the program is no longer operating over two unlike-dimensions to run Gates, which contracts a matrix into one of its indeces without modifying the rest [according to docs](https://quimb.readthedocs.io/en/latest/autoapi/quimb/tensor/tensor_core/index.html#quimb.tensor.tensor_core.Tensor.gate). It is also general now to the PEPS method. The following are left in common between the tutorial and new class, CircuitPEPSSimpleUpdate(Circuit).
```
TN_from_sites_product_state({site: zero for site in self._sites})
gate_simple_()
gauge_all_simple_(gauges=self.gauges, **opts)
```
Some new ones
```
gen_unique_edges(edges)
compute_local_expectation_cluster()
```
The original demo script will probably be deleted to demonstrate the PEPS quantum circuit class, so I'll go through it to preserve its methodology. It also allows enough insertions from the basics to advertise the tensor methodologies quimb can provide.

In [ ]:
Lx = 11
Ly = 11

edges = qtn.edges_2d_square(Lx, Ly, cyclic=False)
sites = sorted({s for e in edges for s in e})

# heisenberg interaction
h2 = -qu.pauli("X") & qu.pauli("X")
# quenched random onsite z-fields
h1 = -3.04438 * qu.pauli("Z")
# all zero-state
psi = qtn.TN_from_sites_product_state(
    site_map={site: [1.0, 0.0] for site in sites}
)
# make size 1 lattice bonds
for cooa, coob in edges:
    psi[cooa].new_bond(psi[coob])
psi
# create simple update gauges
gauges = {}
psi.gauge_all_simple_(gauges=gauges)

results = {}
# truncation options
max_bond = 16
cutoff = 1e-6
t = 0.0
dt = 0.04
U2 = ar.do("linalg.expm", -1j * dt * h2)
U1 = ar.do("linalg.expm", -1j * dt * h1)

mag_terms = {(site,): qu.pauli("Z") for site in [(Lx // 2, Ly // 2)]}
cluster_sets = {c: tuple(psi.gen_gloops_sites(c)) for c in [0, 4, 6]}

[len(cs) for cs in cluster_sets.values()]

def measure():
    if "times" in results and results["times"][-1] == t:
        # already measured this time
        return
    results.setdefault("times", []).append(t)
    for c, gloops in cluster_sets.items():
        results.setdefault(("norm", c), []).append(
            psi.norm_gloop_expand(
                gauges=gauges,
                gloops=gloops,
                optimize="random-greedy",
                autocomplete=True,
                autoreduce=True,
            ).real
        )
    # store some intermediate state for efficiency
    info = {}
    for c, gloops in cluster_sets.items():
        results.setdefault(("mag", c), []).append(
            psi.compute_local_expectation_gloop_expand(
                mag_terms,
                gauges=gauges,
                normalized="prod",
                gloops=gloops,
                optimize="random-greedy",
                autocomplete=True,
                info=info,
            ).real
            / len(mag_terms)
        )

for i in tqdm.trange(10):
    measure()
    for edge in qtn.tnag.tebd.edge_coloring(
        edges,
        "random_sequential",
        group=False,
    ):
        psi.gate_simple_(
            U2,
            where=edge,
            gauges=gauges,
            max_bond=max_bond,
            cutoff=cutoff,
            renorm=False,
        )
    for site in sites:
        psi.gate_simple_(U1, (site,), gauges=gauges)
    # ensure gauge is equilibrated
    psi.gauge_all_simple_(
        max_iterations=1000,
        tol=1e-6,
        gauges=gauges,
        progbar=False,
    )
    t += dt
psi

In [ ]:
for i, c in enumerate(cluster_sets):
    plt.plot(
        results["times"],
        results["norm", c],
        ".-",
        label=f"<psi|psi>C{c}",
        color=mpl.cm.Blues((i + 1) / len(cluster_sets)),
        linewidth=2 * (i + 1) / len(cluster_sets),
    )

for i, c in enumerate(cluster_sets):
    plt.plot(
        results["times"],
        results["mag", c],
        "|-",
        label=f"<M>C{c}",
        color=mpl.cm.Greens((i + 1) / len(cluster_sets)),
        linewidth=2 * (i + 1) / len(cluster_sets),
    )

plt.legend(bbox_to_anchor=(1, 1))
plt.show()
plt.close()

In [ ]:
#relates to text, demonstrates gate function
data = np.array([[[1,0],[0,-1]],[[1,0],[0,-1]]])

nums = qtn.Tensor(data=data, inds=("a","b","c"))
print("Your starting data:")
print(nums.data)
new_tens = nums.gate(qu.pauli('X'), ind='b')
print("After a traditional row-level application:")
print(new_tens.data)
print("Your data is untouched")
print(nums.data)
nums.gate_(qu.pauli('X'), ind='b')
print("Gate in-place:")
print(nums.data)

*   **quimb.tensor.geometry.edges_2d_square(Lx, Ly, cyclic=False)**

Generates a list of tuples, describing edges across Lx and Ly, without a periodic boundary condition. These are 220 elements of((int,int),(int,int))

The sites of the next line does sorted() on a list comprehension, which organizes integers in ascending order. This is done on the s in e, where e is sourced from edges where s is a single inner tuple and e is the outer tuple, the end effect is to place all the inner tuples in order with the second of (int,int) counting up and rolling over the first element.

*   **h2 = -qu.pauli("X") & qu.pauli("X")**

I verified this in the documentation linked, in quimb, bitwise-& like this maps to a kronecker product.

*   **psi = qtn.TN_from_sites_product_state(site_map={site: [1.0, 0.0] for site in sites})**

Used in a different tutorial, also for PEPS with the same site map. Site maps creates a dictionary with the key equal to a tuple in sites, with pointing to \[1.0,0.0\] each. These are identified as product states, and in other versions of the function, are each states assigned to every edge.

*   **for cooa, coob in edges: psi[cooa].new_bond(psi[coob])**

new_bond can be used to create any bond as desired. In this case, it took the tuples of psi and created bonds between them. Any tensor's contents can be called with psi\[index,index\].data. Each contains as its only data \[1.0,0.0\] confirming the state has been assigned to each site with an edge between all of them now.

* **psi.gauge_all_simple_(gauges=gauges)**

This is the part that I find most interesting, the method is to create random connections across all the tensors. It actually creates gauges, which basics tutotial 1.5.1 identifies as placing an operator from a tensor and contracting it unto another tensor without changing properties of the source tensor. In practice, this probably looks like adding elements down the matrix, the QR decomposition is a matrix problem for A = QR, and calling it in the source tensor as $\mathbb{I}$, for example. I have included a link to MPS in Pennylane, which is another example provided on the page, where a series of open bonds become single matrix products that only have those dangling bonds.

*   **ar.do("linalg.expm", -1j * dt * h2)**

autoray is another of jcmgrey's projects, used to sort data types on various backends. In this case, it's adapting the object to do a matrix exponential on h2, of angular multiplier timestep*$i$, the imaginary number.

*   **mag_terms = {(site,): qu.pauli("Z") for site in [(Lx // 2, Ly // 2)]}**

Is divided by its length in a different step, but is only of length 1. Its only element is (5,5) in this case, and it stores a pauli Z. This computes an expectation value of Z at (Lx,Ly) = 5 with generalized loops. In other words, it uses loops with edges bearing at least two connections to create an expectation value from the most symmetric part of the tensor.

*  **cluster_sets = {c: tuple(psi.gen_gloops_sites(c)) for c in [0, 4, 6]}**

Dictionary with entries 0, 4, and 6. gen_gloops_sites generates all such loops where every node is connected to at least two other loop nodes. The argument it takes is the maximum amount of nodes permited in the gloop. In this case, the values are 0, 100, 280.

Personally, this was the most useful function because it identifies this kind of connectivity where it exists in a tensor. For example, each of these would bear a row vector if you tried to compress over an axis.

*   **psi.norm_gloop_expand(gauges=gauges,gloops=gloops optimize="random-greedy", autocomplete=True, autoreduce=True,).real**

This is done in a setdefaults structure, which looks up (norm,c) in the results dictionary to see if it exists. When it doesn't, it appends into a list the result of norm_gloop_expand. This is the frobenius norm, noted in the docs to explicitly be the sum of of the squared singular values across a partition, square rooted. It does it over the randomly formed gauges, using the gloop size specified by the dictionary key, optimized in ordering by a random greedy algorithm. Further details on the value of gathering terms by optimize can be found in section 2 of Tensor network basics on contractions. These keywords are only determining what and how to take the norm operation. Documentation of the autocomplete and autoreduce keywords can be found [here](https://quimb.readthedocs.io/en/latest/autoapi/quimb/tensor/tnag/core/index.html#quimb.tensor.tnag.core.TensorNetworkGenVector.local_expectation_gloop_expand).

The norm issue, of taking an operation along a single axis, should be considered the normal operation in quimb. For example, .H creates the hermitian adjoint, but it might be difficult to conceptualize along what set of axes a transposition is taking place. The answer is that .H only effectively takes the conjugate of every element, as is noted in basics section 1.2. A point is also made that dimensions are not changed, and that T1.H @ T2 is always the frobenius norm, which aligns with this. Be wary that docs/matrix/matrix-basics.ipynb does identify operations where it behaves, particularly in well known states, as expected.

*   **psi.compute_local_expectation_gloop_expand(mag_terms, gauges=gauges, normalized="prod", gloops=gloops, optimize="random-greedy",autocomplete=True, info=info,).real**

Again, mag_terms is only, (5,5):pauli.Z and gloops is each of 4,6 containing loops. This is adding those contractions into a back-up dictionary so recomputation does not occur. This computes expectation value across the loop.

To expand on the usefulness of norm_gloop_expand and compute_local_expectation_gloop_expand, consider the following operator system,

$$\begin{bmatrix} \hat{X} & \hat{Y} \end{bmatrix}
\begin{bmatrix}\hat{X}\\ \hat{Y}\end{bmatrix} = \hat{X}\hat{X} + \hat{Y}\hat{Y}$$

We can describe the outcome as zero dimensional, a single number, if the operators are matrices. However, within each product, there is a fixed row-column relationship. If then the rows of the right matrix are used in a different calculation, you must propagate the individual multiples columnwise. This is why I identified each element as seeing a row-vector. Identifying how and where to pass these on is the value of gloop. Norm carries the traditional magnitude, $(\Sigma \vec{v}^2)^{(1/2)}$ to each, while local expectation actually carries out the $V^{T}W$ into another system.

$$\begin{bmatrix} 0 & 1 \\ 1 & 0 \end{bmatrix}\begin{bmatrix} 0 & 1 \\ \ 1 & 0 \end{bmatrix} = \begin{bmatrix}0(0)+1(1) & (xx)_{01} \\ (xx)_{10} & (xx)_{11}\end{bmatrix}$$

*   **for edge in qtn.tnag.tebd.edge_coloring(edges, "random_sequential", group=False,):**

Classifies the connectivity of the graph such that no two vertexes have the same color if they share an edge. An important feature is that it is about to be used to apply the gates only on unique gauges that perserve the 'only-to-two' motif gloops use. tqdm is a loading bar module ("taqadum" is progress in arabic, "te quiero demasiado" is I love you too much, spanish).

```
psi.gate_simple_(U2, where=edge, gauges=gauges, max_bond=max_bond cutoff=cutoff, renorm=False,)
```
The underscore is used to apply operations in place instead of creating a copy in memory. The most accurate summary of gate_simple, minding object calls, is Tensor.gate. This keeps an indeces object unchanged and produces an array assigned to a novel tensor object. I have linked the class in which gate_simple exists, and it compresses tensors behind it. It applies a simple update, which is to assume things like $\mathbb{I}$ existing in all tensors in its neighbourhood.

*   **for site in sites: psi.gate_simple_(U1, (site,), gauges=gauges)**

Applies gates to vertices, no longer according to graph-coloring. This is Z, wheras the other was adjacency X in the Ising model. I have linked an IBM page explaining the value of these types of computation.

*   Closing the measure loop:

```
    # ensure gauge is equilibrated
    psi.gauge_all_simple_(
        max_iterations=1000,
        tol=1e-6,
        gauges=gauges,
        progbar=False,
    )
    t += dt
```
This once more makes random all connections in the network and adds a timestep.

Predictably, the difference between the script provided and the new CircuitPEPSSimpleUpdate(Circuit) class is use of gen_unique_edges, which performs the same function as graph colouring. It is actually more a gauge function, and in fact makes MPS graph edges. The function compute_local_expectation_cluster includes the location of locality I used in explaining the code, and takes arguments to define where to run calculations over and what to assume outside itself.

---

**Resources**

qu.kron is bitwise-and on \(a & \(b & \(c & ...\)\)\) [quimb.core.kron](https://quimb.readthedocs.io/en/latest/autoapi/quimb/core/index.html#quimb.core.kron)

In relation to gauges, Matrix-Product States are an example provided on the quimb basics page [Pennylane/demos/tutorial_mps](https://pennylane.ai/demos/tutorial_mps)

qtn.norm is the frobenius norm [quimb docs - norm](https://quimb.readthedocs.io/en/latest/autoapi/quimb/tensor/tensor_core/index.html#quimb.tensor.tensor_core.Tensor.norm)

.H@ always produces the Frobenius norm [Excerpt](https://quimb.readthedocs.io/en/latest/tensor/tensor-basics.html#creating-tensor-networks)

The pre-included bell, chiral, etc states will react normally to .H [docs/matrix/matrix-basics.ipynb](https://github.com/jcmgray/quimb/blob/0ccbee272f3eda408fc45f7db75dcffcc34c6512/docs/matrix/matrix-basics.ipynb#L181)

Basics on Contractions, Section 2 of Tensor Network Guide [quimb docs](https://quimb.readthedocs.io/en/latest/tensor/tensor-contraction.html)

In LatticeBondMap, tensor_network_ag_gate is a wrapper for gate operations [quimb docs](https://quimb.readthedocs.io/en/latest/autoapi/quimb/tensor/tnag/core/index.html#quimb.tensor.tnag.core.tensor_network_ag_gate)

The next entry in the LatticeBondMap class is gate_simple [quimb docs](https://quimb.readthedocs.io/en/latest/autoapi/quimb/tensor/tnag/core/index.html#quimb.tensor.tnag.core.tensor_network_ag_gate_simple)

IBM coverage of the Ising Model titled "Two Interacting Magnets" in the Qiskit "Use a Quantum Computer Today" online module. No account needed. [Qiskit- your first quantum experiment](https://quantum.cloud.ibm.com/learning/en/courses/use-a-qc-today/your-first-quantum-experiment)

compute_local_expectation_cluster contains definitions of locality to produce a reduced density matrix. Great summary of the goals set out by the script [quimb docs](https://quimb.readthedocs.io/en/latest/autoapi/quimb/tensor/tnag/core/index.html#quimb.tensor.tnag.core.TensorNetworkGenVector.compute_local_expectation_cluster)

##$|$Toqito$\rangle$ Bounty ($100): Implement channel exclusion (channel antidistinguishability) (\#1521)
*Awarded to: mathTar for Pull # 1570* [Direct](https://github.com/vprusso/toqito/pull/1570)

$|$Toqito$\rangle$ is a quantum information package with simple syntax that provides easy access to many famous experiments. This bounty is exemplary as essentially taking quantum data from deeper within the package to run through a popular optimizer package, picos. Recent implementations have focused on antidistinguishability, which is the process of rejecting a state based on observables. Supporting matrix work, it is headed by member of the Unitary Foundation, and is expected to continue recieving physics-informed updates.

Won by a Texas A&M graduate student, it has high-quality doc strings detailing the steps the program takes and is worth a read. The channel exclusion function is an if-loop around what numerical strategy to use. I've collected the parts that feed variables into further functions.

```
#in toqito/channel_metrics/channel_exclusion.py
from toqito.channel_ops import kraus_to_choi
from toqito.channel_props.channel_dim import channel_dim
def channel_exclusion(
    channels: list[np.ndarray | list[np.ndarray] | list[list[np.ndarray]]],
    probs: list[float] | None = None,
    strategy: str = "min_error",
    solver: str = "cvxopt",
    primal_dual: str = "dual",
    **kwargs,
) -> tuple[float, list[np.ndarray]]:
    """
    Args:
    channels: List of channels, each provided as a Choi matrix or as Kraus operators.
    probs: Prior probabilities for the channels. If omitted, a uniform distribution is used
    [...]
    Returns:
    The optimal exclusion probability and a list of optimal strategy operators.
    """
    n_channels = len(channels)
    probs = [1 / n_channels] * n_channels if probs is None else probs
    probs_arr = np.array(probs, dtype=float)
    probs_sum = float(np.sum(probs_arr))
    probs_arr = probs_arr / probs_sum
    choi_channels: list[np.ndarray] = []
    channel_dims = []
    for channel in channels:
        dim_in, dim_out, _ = channel_dim(channel)
        channel_dims.append(np.array([dim_in, dim_out]))
        choi_channels.append(kraus_to_choi(channel) if isinstance(channel, list) else channel)
    first_dim = channel_dims[0]
    dim_in = int(first_dim[0][0])
    dim_out = int(first_dim[1][0])
    if strategy == "unambiguous":
        return _unambiguous_primal(choi_channels, probs_arr.tolist(), dim_in, dim_out, solver=solver, **kwargs)
    if primal_dual == "primal":
        return _min_error_primal(choi_channels, probs_arr.tolist(), dim_in, dim_out, solver=solver, **kwargs)
    return _min_error_dual(choi_channels, probs_arr.tolist(), dim_in, dim_out, solver=solver, **kwargs)
```
channel_dim takes a superoperator input, from which a Kraus is 1d or 2d lists of numpy arrays. That operation unpacks it into channel dim = \[\[dim, dim\], \[dim, dim\]\]. From now on, the choi_channels fed to each solution strategy will be list\[np.ndarray\] matrices. It will solve for the first value of each row in the channel_dim output I showed.

The Hermitian_variable stores each nd.array in a vectorization convenient for solving the system. Its arguments are only a name and dimensionality, it took no data. I have translated the constraints into LaTex. Superscripts denote j of a collection where dimension is shown explicitly. The `>>` is the positive-semidefinite condition such that both $W_j$ and $X$ all have positive or zero eigenvalues.

$$tr(X)=1$$
$$\sum_{j} W_{in*out\times in*out}^{j}= X_{in\times in} \otimes \mathbb{I}_{out}$$
One might be inclined to compare a trace 1 form of an inner system to a quotient optimization, and the entire system resembles relative entropy of entanglement criteria. This quantifies the distance from a seperable, or direct product state, collection of density matrices[1]. The trace condition is directly referenced in Manna & Das Bhowmik, as the sum of all complementary measurements equaling one only leading to matching $M$ and $\rho$ states (c.f. eq 6) such that X would be matched to an $M_k$ by the complementary space (c.f. eq 2) [2].

$$E_R(\rho)=\min_{\sigma \in D}\mathbb{tr}\big(\rho \mathbb{log}(\rho) - \rho \mathbb{log}(\sigma)\big)$$

The objective is the minimization of inner product of the Choi matrix with the strategy operator, weighed by the probability of the system being in that state. Manna & Das Bhowmik implement this in equation 2 from equation 1 as referencing the sum of probabilities achieving this state to be 1, which is why it is a relative entanglement measurement after rearranging the constraint unto one side, multiplying trivially by 1 and maximizing complementary probabilities.

$$A[\{V_j\}_j,\{p_j\}_j]=\min\big(\sum_jp_j\mathbb{vec}(V_{j}^{\dagger})\mathbb{vec}(W_j)\big)$$

The equation returns the antidistinguishability probability distribution [2]. In a tuple, it also includes the superoperator that can be multiplied with input state V for the antidistinguishable measurement.
```
#in toqito/channel_metrics/channel_exclusion.py
import picos as pc
def _min_error_primal(
    channels: list[np.ndarray],
    probs: list[float],
    dim_in: int,
    dim_out: int,
    solver: str = "cvxopt",
    **kwargs,
) -> tuple[float, list[np.ndarray]]:
    n_channels = len(channels)
    problem = pc.Problem()
    strategy_ops = [
        pc.HermitianVariable(f"W[{idx}]", (dim_in * dim_out, dim_in * dim_out))
        for idx in range(n_channels)
    ]
    x_var = pc.HermitianVariable("X", (dim_in, dim_in))
    problem.add_list_of_constraints(strategy_ops[idx] >> 0 for idx in range(n_channels))
    problem.add_constraint(x_var >> 0)
    problem.add_constraint(pc.trace(x_var) == 1)

    # Choi operators are ordered as input x output, so the lifted marginal is X x I_out.
    problem.add_constraint(pc.sum(strategy_ops) == x_var @ np.eye(dim_out))

    objective = pc.sum([probs[idx] * (channels[idx] | strategy_ops[idx]).real for idx in range(n_channels)])
    problem.set_objective("min", objective)
    solution = problem.solve(solver=solver, **kwargs)
    return solution.value, [np.array(var.value) for var in strategy_ops]
```
The dual of a minimization is a maximization. Interestingly, the y variable must be smaller than the probability of the channels. A partial trace over Y in the output dimension must also be larger than a real number.
$$p_jW_j \succeq Y$$
$$\mathbb{tr}_{1 =(out \times out)}(Y_{in*out \times in*out}) \succeq \lambda \mathbb{I}_{in}$$

The implementation of partial trace is unambiguous as system $A_0 \otimes A_1 \otimes ... A_{n-1}$ will return $B_0 \otimes ... \otimes B_{n-1}$ of $B_i = tr(A_i)$. The objective is then to maximize the partial trace
$$A[\{V_j\}_j,\{p_j\}_j] = \max(\lambda \in \mathbb{R})$$
```
#in toqito/channel_metrics/channel_exclusion.py
import picos as pc
def _min_error_dual(
    channels: list[np.ndarray],
    probs: list[float],
    dim_in: int,
    dim_out: int,
    solver: str = "cvxopt",
    **kwargs,
) -> tuple[float, list[np.ndarray]]:
    n_channels = len(channels)
    problem = pc.Problem()
    y_var = pc.HermitianVariable("Y", (dim_in * dim_out, dim_in * dim_out))
    lambda_var = pc.RealVariable("lambda")
    dual_constraints = [problem.add_constraint(y_var << probs[idx] * channels[idx]) for idx in range(n_channels)]
    problem.add_constraint(pc.partial_trace(y_var, 1, (dim_in, dim_out)) >> lambda_var * np.eye(dim_in))
    problem.set_objective("max", lambda_var)
    solution = problem.solve(solver=solver, **kwargs)
    strategy_ops = [np.array(constraint.dual) for constraint in dual_constraints]
    return solution.value, strategy_ops
```
Finally, it can be solved by Semidefinite Programming. This relies on all solution variables being antidistinguishable with one state, thus making the state complete, and making it complementary. The biggest difference between this and the primal problem is that there is an $W_{inc}$ that can be added to the sum to equal that $X$ condition.

$$\mathbb{vec}(V_j^{\dagger})\mathbb{vec}(W_j) = 0$$

$$W^{inc} + \sum_{j}W^j = X_{in \times in} \otimes \mathbb{I}_{out}$$

The objective is to minimize overlap with such a $W_{inc}$

$$A[\{V_j\}_j,\{p_j\}_j] = \min\big(\sum_jp_j\mathbb{vec}(V_j^{\dagger})\mathbb{vec}(W_{inc})\big)$$
```
def _unambiguous_primal(
    channels: list[np.ndarray],
    probs: list[float],
    dim_in: int,
    dim_out: int,
    solver: str = "cvxopt",
    **kwargs,
) -> tuple[float, list[np.ndarray]]:
    n_channels = len(channels)
    problem = pc.Problem()

    W_vars = [pc.HermitianVariable(f"W[{i}]", (dim_in * dim_out, dim_in * dim_out)) for i in range(n_channels)]
    W_inc = pc.HermitianVariable("W_inc", (dim_in * dim_out, dim_in * dim_out))
    x_var = pc.HermitianVariable("X", (dim_in, dim_in))

    problem.add_list_of_constraints(W_vars[i] >> 0 for i in range(n_channels))
    problem.add_constraint(W_inc >> 0)
    problem.add_constraint(x_var >> 0)
    problem.add_constraint(pc.trace(x_var) == 1)

    problem.add_constraint(pc.sum(W_vars) + W_inc == x_var @ np.eye(dim_out))

    # Zero-error constraints for conclusive outcomes
    problem.add_list_of_constraints((channels[i] | W_vars[i]) == 0 for i in range(n_channels))

    objective = pc.sum([probs[i] * (channels[i] | W_inc) for i in range(n_channels)])
    problem.set_objective("min", objective)

    solution = problem.solve(solver=solver, **kwargs)

    return solution.value, [np.array(var.value) for var in W_vars] + [np.array(W_inc.value)]
```


---
**Resources**
*Disclosure:* The pipes notation is not so clearly found in the documentation, and I needed Google Gemini to translate that expression for me.

The \_\_or\_\_ picos solver adaption for pipe operators representing Frobenius inner product. On the same page, partial_trace including theoretical representation: [picos.expressions.exp_affine](https://picos-api.gitlab.io/picos/api/picos.expressions.exp_affine.html)

Qiskit implementation of Choi matrices, to introduce the type of problem being solved here [Qiskit Docs](
https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.quantum_info.Choi)

More consise summary of the partial trace than offered by Wikipedia, by a Waterloo professor whose work is referenced by the bounty's description [Watrous Lecture Notes - Partial Trace](
https://cs.uwaterloo.ca/~watrous/QC-notes/QC-notes.15.pdf)

A useful resource for working with Hilbert operators and superoperators in the basic case of the Liouville space, since I mentioned it by name! [Gyafi- Fundamentals of QM in Liouville space](https://arxiv.org/abs/2003.11472)

---
References

[1]. Williams, C.P. Quantum Information. *Explorations in quantum computing*, 2nd ed.; Springer London, 2011; pp 403-482. DOI: [https://doi.org/10.1007/978-1-84628-887-6](https://doi.org/10.1007/978-1-84628-887-6)

[2]. Manna, S.; Das Bhowmik, A. Single-shot Antidistinguishability of Unitary Operations. *Physical Review A* **2026**, 113(2), 022218. DOI:https://doi.org/10.1103/d183-k1x3 ArXiv:2510.14609


##Extension: *Single Shot Anti-Distinguishability of Unitary Operations,* by Manna & Das Bhowmik

This paper's work is most directly implemented in the primal implementation, as the reals condition, to the minimization. I would like to present their cosines framework for entanglement, resummarizing the paper in a concise capacity. This work only applies to three simultaneous gates acting on two dimensional Hilbert space, as the conditions change in higher dimensions.

Defining the space, antidistinguishability is a probability of previously-known state $\{\rho_{k}\}^{n}_{k=1}$ sampled in probability distribution $\{q_k\}_{k=1}^{n}$. Already, we can work out the following standard manipulation from projective measurements. I have linked a Pennylane resource that explains it at a good level.

$$M[\rho] = \sum^{n}_{k=1}p_{k}\rho_{k}=\sum_{k=1}^{n}M_{k}\rho M_{k}$$
$$M_k=|k\rangle\langle k|$$
$$M[\rho] = \sum_{k=1}^{n}|k\rangle\langle k|\rho|k\rangle\langle k|$$
$$\rho_k = \frac{M_k\rho M_k}{\text{Tr}(M_k\rho)}$$
$$p_k = \text{Tr}(|k\rangle\langle k|\rho)=\sum_{a=1}^{n} \langle k |a \rangle \langle a|k\rangle \Rightarrow \text{max}(p_k) = \langle k |k \rangle \langle k|k\rangle$$

In this work, measuring in the incorrect basis is responsible for propagation of state deeper into a computer, the essense of quantum computing. In presenting fraction $\{q_k\}_{k=1}^{n}$ of outcomes, the simplification for probability and maximization is carried out by the paper.
$$A[\{\rho_k\}_k,\{p_k\}_k]=\max_{\{M_a\}_a}\Big(\sum_{k,a}q_kp(a\neq q|\rho_k,M_a)\Big)$$
$$A[\{\rho_k\}_k,\{p_k\}_k]=1-\min_{\{M_k\}_k}\Big(\sum_{k}q_k \text{Tr}(\rho_kM_k)\Big)$$

The following necessary and sufficient conditions antidistinguish three pairwise non-orthogonal quantum states [2].

$$g_1+g_2+g_3 < 1$$
$$(g_1+g_2+g_3-1)^2 \geq 4g_1g_2g_3$$
$$g_1=|\langle \psi_1|\psi_2\rangle|^2,g_2=|\langle \psi_2|\psi_3\rangle|^2,g_3=|\langle \psi_1|\psi_3\rangle|^2$$

Another sufficient condition for n states is as follows, also presented with the n-1 simplification, and presented for the present system.

$$|\langle \psi_i | \psi_j \rangle| \leq \frac{1}{\sqrt{2}}\sqrt{\frac{n-2}{n-1}}= \frac{1}{\sqrt{2}}\sqrt{1+\frac{1}{n-1}}$$

$$1 \leq i\ne j \leq n$$

$$|\langle \psi_i | \psi_j \rangle| \leq \frac{1}{\sqrt{2}}\sqrt{\frac{3-2}{3-1}}=\frac{1}{2}$$

Spectral decomposition of $U_q^{\dagger}U_j = \sum_{l=1}^{d}e^{i\theta_{qj}^{l}}|\psi_{l_{qj}}\rangle\langle \psi_{l_{qj}}|$ of ascending eigenvalue l of value $e^{i\theta^{l}_{qj}}\in\mathbb{C}$ in the product qj is carried out for a non-maximally entangled direct product state. $i$ is the imaginary number $\sqrt{-1}$. Every $\psi$ in the following expectation value can be decomposed into $|\psi\rangle = \sum_{l=1}^{d}\alpha_{l_{ij}}|\psi_{l_{ij}}\rangle$ corresponding to the eigenvector components from $\psi$ for eigenvalue l.
$$|\langle \psi|U^{\dagger}_qU_j|\psi\rangle|^2=\Bigg|\sum_{l=1}^{d}|\alpha_{l_{qj}}|^2e^{i\theta_{l}^{qj}}\Bigg|^2$$
Forming a set of complex numbers made up of convex combinations of $\{e^{i\theta_{qj}^{l}}\}$. This product is of interest from the cyclical permutation property of the trace.

$$\big(Tr(U|\psi\rangle\langle\psi|U^{\dagger})\big)^2 = \big(Tr(\langle\psi|U^{\dagger}U|\psi\rangle)\big)^2$$

Lower registers to the application of a gate represent a degree of redundancy in the basis and are ignored in the following work on maximally entangled probing state $|\Phi\rangle = \frac{1}{\sqrt{d}}\sum_{k=1}^d|kk\rangle$.
$$|\langle \Phi|U^{\dagger}_qU_j|\Phi\rangle|^2 = \frac{1}{d^2}\Bigg|\sum_{k=1}^{d}\langle k| U_q^{\dagger}U_j|k\rangle\Bigg|^2$$
$$= \frac{1}{d^2}\Bigg|\text{Tr}(U_q^{\dagger}U_j)\Bigg|^2=\frac{1}{d^2}\Bigg|\sum_{l=1}^{d}e^{i\theta_{qj}^{l}}\Bigg|$$
The spectral decompositions are useful for the following two models, one of a non-maximally entangled probe, and another of a maximally entangled state, respectively. In taking the squares, only the cosine terms of the eigenvalues survive. The convex condition is that the eigenvalues have weights summing up to 1, represented by t. The labelling of the unitaries follows the numering presented at the beginning of the section.
$$g_{1_{NM}}=|te^{i\theta_{12}^{1}}+(1-t)e^{i\theta_{12}^{2}}||te^{-i\theta_{12}^{1}}+(1-t)e^{-i\theta_{12}^{2}}|$$
$$=t^2+(1-t)^2+2t(1-t)cos(\Theta_{12}^{1}-\Theta_{12}^{2})$$
Note that the following expression from the paper uses $0=(1/2)cos(\Theta_{12}^1-\Theta_{12}^2)-(1/2)cos(\Theta_{12}^1-\Theta_{12}^2)$ in the following factored form, which the authors included to fascilitate comparison in the first term to the maximally entangled form.

$$g_{1_{NM}}=\frac{1+cos(\Theta^1_{12}-\Theta^2_{12})}{2}+2(t-\frac{1}{2})^2\big(1-cos(\Theta_{12}^{1}-\Theta_{12}^{2})\big)$$

Because the angle differences are simply phase shifts, this equation contains no negative terms. This important observation is the reason cited in the paper for why a non-maximally entangled condition has a higher value than the maximally entangled one, and therefore any passing condition in a non-maximally entangled state automatically passes with an entangled one [2]. To demonstrate the trivial distinguishable exclusions, it is easy to observe, using the same 1 = (1/2) + (1/2) method as the authors, that $cos(\Theta_{12}^{1}-\Theta_{12}^{2})=1$ creates $g_{1_{NM}}=0$ which fails the condition.

$$g_{1_{NM}}=2(t^2-t+\frac{1}{4})+(2t-2t^2)cos(\Theta_{12}^{1}-\Theta_{12}^{2})+\frac{1}{2}$$
$$=1$$

And $cos(\pi/2)=0$ is a trivial comparison between parabolas.

$$g_{1_{NM}}=t^2-2t+1$$

In the intermediate values, the following rearrangement of the inner product might prove more useful.

$$2t^{2}(1-cos(\Theta_{12}^{1}-\Theta_{12}^{2}))-2t(1-cos(\Theta_{12}^{1}-\Theta_{12}^{2}))+1$$

The maximally entangled product for d=2 is easier, and clearly admits negative values.

$$g_{1_M}=\frac{1}{4}\Bigg|e^{i\Theta_{12}^{1}}+^{i\Theta_{12}^{2}}\Bigg|^2$$
$$g_{1_M}=\frac{1}{2}+\frac{1}{2}cos(\Theta_{12}^{1}+\Theta_{12}^{2})$$

As a word sentence, the number from subtracting 1 can be greater in M than NM, but NM must pass its condition, which is smaller than the M values.

$$(g_{1_M}+g_{2_M}+g_{3_M}-1)^2 \geq (g_{1_{NM}}+g_{2_{NM}}+g_{3_{NM}}-1)^2 \geq 4g_{1_{NM}}g_{2_{NM}}g_{3_{NM}} \geq 4g_{1_{M}}g_{2_{M}}g_{3_{M}}$$

This code block demonstrates intermediate admissible values for non-maximally entangled probes.

---
**Resources**

Beginner-friendly Pennylane page on mid-circuit measurements, or projective measurements: [https://pennylane.ai/demos/tutorial_mcm_introduction](https://pennylane.ai/demos/tutorial_mcm_introduction)

---
References

[2].Manna, S.; Das Bhowmik, A. Single-shot Antidistinguishability of Unitary Operations. *Physical Review A* **2026**, 113(2), 022218. [DOI:https://doi.org/10.1103/d183-k1x3](https://doi.org/10.1103/d183-k1x3) [ArXiv:2510.14609](https://arxiv.org/abs/2510.14609v2)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

SAMPLES = 99

t = np.linspace(0,1,SAMPLES,dtype=float)
a = np.linspace(0,2,SAMPLES,dtype=float)
x = np.zeros((SAMPLES,SAMPLES))
for num in range(0,SAMPLES):
    for numb in range(0,SAMPLES):
      x[num,numb] = a[num] * t[numb]
x.reshape(-1)
y= np.zeros((SAMPLES,SAMPLES))
for num in range(0,SAMPLES):
    for numb in range(0,SAMPLES):
      y[num,numb] = 2*a[num]*(t[numb]**2)-2*a[num]*(t[numb])+1.0
y.reshape(-1)
fig,ax = plt.subplots()
ax.scatter(x,y)
ax.set_title("Cosine/t signing versus criterion value")
ax.set_ylabel("<1 value")
ax.set_xlabel("(1-cos) * t magnitude and sign")
plt.show()
print(y[y<0.5].shape)

##Background: Paulie's Pauli String Lie Groups
I completed simple arithmetic to prove the nature of identities in the appendix of Aguilar et. al. 2024 around the antisymmetry of certain Pauli strings, including the case of paulis that end in Y. The paper is structured so well that it is best to follow its arguments, side-proofs, and conclusions in order to demonstrate how to read such a work. I start particularly by noting that the graph theory notation the paper opens with is deferred for a future finder algorithm, but *Paulie* has already implemented this functionality around adjacency matrix row-column relationships. We must use the set-theory arguments only.

The motivation is, given the operation $i[P_m,P_n] = i(P_mP_n - P_nP_m)$ of the *m*-th element of a pauli string with the *n*-th element of a pauli-string $...\otimes P_m (...\otimes_{0<j<n-1}...)P_{n}\otimes...$ where P is a pauli matrix, how many times can this operation be applied such that the remaining pauli string will shrink its span if any one element is removed? An important observation is that $[P_mP_n] = \pm 2P_{m}P_{n}$ which is another pauli, and that any pauli that does not commute excluding Identity will strictly anticommute. This is the basis for the graphical proofs of the paper around belonging as seeking out where anti-commutation is possible, and writing it as mutual exclusivity with a contraction. Then that contraction must have a complementary relationship with its outcome because all connecting matrices have some common element. Where $m$ and $n$ are being used as placeholders for paulis in a sequence, we now adopt the convention $\sigma_{i}$ which is pauli matrix $\sigma \in {X,Y,Z}$ pauli matrix applied on the *i*-th qubit making it more obvious that shrinkage is based on $X,Y$ occupying the same qubit as $Z$ in the cyclic permutation of the commutation relationships.

To illustrate the background paper's point. Consider the spanning set of qubit states making up X and Z.

$$Z = |0\rangle \langle 0| - |1\rangle \langle 1|$$

$$X = |1\rangle \langle 0| + |0\rangle \langle 1|$$

Noting that a hilbert space exists in this set,

$$\mathbb{I} = \sum^{1}_{\phi=0}|\phi\rangle\langle \phi| = |0\rangle \langle 0| + |1\rangle \langle 1|$$

The following can be observed about the relationship of on-diagonal and off-diagonal states in the set.

$$Z = (\mathbb{I})Z(\mathbb{I}) = \sum^{d}|\phi\rangle \langle \phi |\big(|0\rangle \langle 0 |\big)\sum^{d}|\phi\rangle \langle \phi | - \sum^{d}|\phi\rangle \langle \phi |\big(|1\rangle \langle 1 |\big)\sum^{d}|\phi\rangle \langle \phi |$$

$$Z = (|0\rangle\langle 0|)_{0,0} + (|1\rangle\langle 1|)_{1,1}$$

$$\langle Z \rangle \in \langle 1|Z|1 \rangle + \langle 0|Z|0 \rangle = 0$$

Is mutual exclusivity in the trace, and for off-diagonal elements using the identical $\mathbb{I}( X )\mathbb{I}$ form,

$$X=(|0\rangle\langle1|)_{0,1} + (|0\rangle\langle1|)_{1,0}$$

$$\langle X \rangle \in \langle 1|X|1 \rangle + \langle 0|X|0 \rangle = 0$$

$$(\mathbb{I})X(\mathbb{I}) \in \mathbb{span}(\{X,Z\}) \ne 0$$

Is to say that any pauli string is spanned by orthogonal states 0, 1 where simultaneous basis states 0,1 cannot exist at the same time in any combination. Any mixed density state is contained in the span of these pauli matrices as not empty, but similarly traceless. Each qubit is limited to commuting within X or Z with Y, conforming to the symplectic form, to be covered later. Swapping in the cyclically permutating commutator makes this obvious, since including any other description of the qubit will anti-commute unevenly by $[\sigma_1,\sigma_2] = -[\sigma_2,\sigma_1]$. In Appendix E of the source, by a direct sum. In the following notation, $\{0,1\}^{n_c}$ means all registers up to $n_c$, and is just an eigenstate written in binary.

$$|\vec{i}\rangle\langle\vec{i}| \otimes P_{\mathfrak{g}}$$

$$\vec{i} \in \{0,1\}^{n_c}$$

$$\big\{\bar{P} \otimes \mathbb{I}_{N}\big\}\oplus \big\{\bar{P} \otimes Z_N\big\}$$

$$Z_{1}^{i_1} \otimes ... \otimes Z_{n_c}^{i_{n_c}}\otimes P_{\mathfrak{g}}$$

This creation of subsets is made explicit in the following work.

The subset $S_i$ is an observation on the mutual exclusivity between commuting and anti-commuting properties of any qubit level of a pauli string.

$$S_i = \{P\in S |P^{T}Q+(-1)^{i}QP=0\}$$

$$i \in \{0,1\}$$

For $L = V \cdot W$, $V \in S_{i_V}$ and $W \in S_{i_W}$, the following relationship proves that membership of $S_0$ is evidence of $[V,W]_{+}=0$. If have undertaken arithmetic to show the math hidden for this identity to demonstrate that this is no special Q, which appears in the margins as progressive application of the subset identity. Therefore, the first line is to show the motivation of writing a (-1)(-1) form without an antisymmetric Q necessity. Then work is presented to arival at the $PQ$ form through transposition and antisymmetric L resulting from V. Then the work from the paper is presented as it appears, where then the W set influences the final identity. $i_{S_V}$ and $i_{S_W}$ are not symbols for repeated application by the length of the pauli string, they demonstrate the (-1) term from belonging to a subset. It is easy to use the first line to demonstrate that if both are symmetric, the individual (-1) of the subsets formula would cancel across the second block's $(-1)^{i+1}$.

$$W^TV^TQ = -1W^TVQ = -W^T(Q^TV^T)^T = (-1)(-1)W^T((V^TQ)^T)^T) = (-1)(-1)W^TV^TQ$$

$$P^TQ = (-1)^{i+1}QP$$

$$(Q^TP)^T=(-1)^{i}(PQ^T)^T$$

$$PQ = (-1)^{i+1}QP^T$$

$$L^TQ \pm QL = (VW)^TQ \pm QVW = W^TV^TQ \pm QVW = (-1)(-1^{i_V})W^TQV \pm QVW$$

$$=(-1)^2(-1)^{i_V + i_W}QWV \pm QVW = Q((-1)^{i_V+i_W}WV \pm VW)$$

Note the product $P_{C_{1}}P_{C_{2}}\in S_{1}$ in the paper, $[P_{C_1},P_{C_2}] = 0$ is true.

It is productive to now get a landscape for the different groups the bounty wants to work with: $\mathfrak{so}(n_L+1)$, $\mathfrak{sp}(2^{n_2})$, $\mathfrak{so}(2^{n_2+3})$, $\mathfrak{su}(2^{n_2+2})$. The numbers in the brackets represent the dimensionality of the final matrix itself, and elements will be direct summed up to that. In fact, the desired solution is marginally related to Paulis themselves and is more related to relationships between elements. This work is to demonstrate that the Paulis are not sparse to that.

$\mathfrak{su}(N)$ is traceless, and becomes $\mathfrak{u}(N)$ when Identity pauli matrices are included in the set. This is because $log(\{P_{i}\}_{i})$ is no longer traceless, and $\mathfrak{su}$ is characterized by a determinant 1 condition, which is a sum of the logarithms of eigenvalues equaling zero.
$$\mathfrak{su}(N)\equiv \{P\in \mathbb{C}| P^{\dagger}=P, Tr(P) =0\}$$
Antisymmetric matrices are a real-valued exception constituting $\mathfrak{so}$ and remain traceless by condition of determinant = 1 in $\mathfrak{su}$.
$$\mathfrak{so}(N)\equiv\{P\in\mathfrak{su}(N)|P^{T}=-P\}\subset \mathfrak{su}(N)$$
In this following expression, there is lower strings than $\Omega$ satisfying the final equation, which revolves mainly around the role of Y. This matches usual definitions of $\mathfrak{sp}$ using a similar constructor.
$$P^{T}\Omega = -\Omega P$$
$$\Omega \equiv \begin{bmatrix} 0 & \mathbb{I}_N \\ -\mathbb{I}_N & 0 \end{bmatrix} = iY_0\otimes \mathbb{I}_N$$
$$\mathfrak{sp}\Big(\frac{N}{2}\Big) = \{P \in \mathfrak{su}(N)|P^TY_0=-Y_0P\} \subset \mathfrak{su}(N)$$

It may be obvious from the previous work that the strings follow the following rule, where C denotes the connected matrices that can be contracted by the Lie operation of commutivity. $n_{CC}$ is the total number of connections.
$$P_{C_1}...P_{C_{n_{cc}}}\in S_{(n_{cc}+1)\mathbb{mod}2}$$
Continuing to a case of $Z_0Z_C$, here a Z exists on a different qubit C to persist by having no other activity to contract it- with a corresponding $Y_i$ on a main branch. It conforms to the identity set out by the symplectic group for this reason, with $K^TQ + QK = 0$ work now having $Q = Y_0$, where $K$ is a pauli string. In that work, P and Q are individual paulis but this detail has been replaced by my work demonstrating no special Q of the identity. Note the subscripted $i$ in the final union exponentiates the number members of single controlling bits connection C to appear off the central $Y_0Z_0$ contraction.

$$S_{B1} = \{Y_0Z_0\}\cup\{Z_0Z_i,Y_i\}_{i=1,...,n_2}\cup\{Z_0Z_C^{i}\}_{i=1,...{n_C}}$$
$$P^TY_0=-Y_0P$$
$$Z_0\cdot Z_0Z_i \cdot Z_i = \mathbb{I}_0\mathbb{I}_i$$

This is perhaps the easiest group to be shown to be a direct product for that reason.

$$span(S_{B1})=\oplus^{2^{n_c}}_{i=1}\mathfrak{sp}(2^{n_2})$$

Now adding a $Y$ for every qubit C causes $Q = \mathbb{I}_{n+1}$, greatly simplifying $\mathfrak{so}$ work. This demonstrates the importance of solving for a general Q in that early work. Note that it incorprates a final $Y_i$, $Y_n$ and expands the lie to $Y_{n+1}$ qubits so that the identity matrix in the $K^T = -K$ expression is the number of length-two pauli strings plus three. The statement $C_i \in I$ can safely be interpreted to mean unity, because the purpose of these control bits is to only create anti-symmetrical Y-matrices.

$$S_{B2}=\{Y_0,Z_0Y_{n+1}\}\cup\{Z_0Z_iY_{n+1},Y_{i}\}_{i=1,...,n_2-1}\cup\{Z_0Z_{C_i}Y_{n+1}\}_{C_i\in I}\cup \{Z_0Z_nY_{n+1},Y_n,Z_nY_{n+1},Y_0X_{n+1}\}$$

$$K^T\mathbb{I}_{n_2+3}+\mathbb{I}_{n_2+3}K$$

$$Z_0Y_{n+1}\cdot Z_0Z_nY_{n+1}\cdot Z_nY_{n+1} = Y_{n+1}$$

$$Z_0Z_iY_{n+1} \cdot Z_0Z_iY_{n+1} = \mathbb{I_0}\mathbb{I}_i\mathbb{I}_{n+1}$$

$$span(S_{B2}) \subseteq \mathfrak{so}(2^{n_2+3})$$

Using figure 3, I will justify this observation. $\{Z_nY_{n+1},Y_n,Z_0Z_nY_{n+1}\}$ is related by qubit n and a $Y_n$ preventing total contraction. Two sets stem from that, $Z_0Z_CY_{n+1}$ is sized by not having a $Y_i$ associated with any C as by non-membership to $\{Z_0Z_iY_{n+1},Y_i\}$ where every i is matched. There is also a $Z_0Y_{n+1}$ that cannot be compressed because of being in the highest register $Y_{n+1}$, lacking lower-register matrices. Thus, any $Z_C$ matching $Z_i$ will take its place by multiplying to identity.

This concludes the set work carried out by the paper. The remaining work they present revolves around mapping the strings to gates, and demonstrating that the Lie group is not interrupted by Clifford operations.

Clifford transformations do not change the Lie algebra [3]. Inserting the matrix form of the gate C, I have furnished some of the paper's work with the appropriate cancellation to make the non-special Q clear. C is a unitary matrix.

$$\mathfrak{g} = \mathbb{span}(\{P | P^TQ + QP = 0 \}$$

$$P \mapsto P'=C^{\dagger}PC$$

$$0 = (CP'C^{\dagger})^TQ + Q(CP'C^{\dagger})$$

$$=C^{*}P'^{T}C^{T}Q+QCP'C^{\dagger}$$

$$=(C^T)C^{*}P'^{T}C^{T}Q(C)+(C^T)QCP'C^{\dagger}(C)$$

$$=P'^{T}C^{T}QC+C^{T}QCP'$$

$$=P'^{T}Q'+Q'P'$$

A mapping $Q\mapsto Q' = C^{T}QC$ is allowed to exist, but Paulis are mapped up to a phase [3]. The antisymmetry/symmetry of Q is preserved by transposition of the map.

$$Q'^{T} = C^TQ^TC$$

The paper presents generating set $(S,H,CNOT)$ as having important mappings, that the reader is highly encouraged to glance at in Appendix F.

Of the traceless Pauli matrices, where the commutator with an upper Y is zero, and the pauli matrix is equal to the negative of its transpose - only true of iY - Y is transformable into every non-identity Pauli. This is a very useful property, but it requires two sequential Y's in a string. The final line follows from the set condition, $P^T = -P$

$$\mathfrak{g} = \mathbb{span}(\{P\in (\sigma_i \otimes ... \otimes \sigma_n \backslash {\mathbb{I}})|[Y_{n+1},P],P^T+P = 0\} $$

$$0 = (CP'C^{\dagger})Y-Y(CP'C^{\dagger})$$

$$0 = (C^{\dagger})CP'C^{\dagger}Y(C) - (C^{\dagger})YCP'C^{\dagger}(C)$$

$$0 = P'C^{\dagger}Y(C) - C^{\dagger}YCP'$$

$$Q:=C^{\dagger}YC$$

$$0 = (CP'C^{\dagger})^T+(CP'C^{\dagger})$$

$$0 = (C^T)C^{*}P'^TC^T(C)+(C^T)CP'C^{\dagger}(C)$$

$$0 = P'^TC^TC+C^TCP'$$

$$K:=C^TC$$

$$0 = -C^TPC^{*}C^TC^+C^TCP'$$

The final implication presented in the paper appears to suggest that real gate sets $C^TC$, which would be their own inverses, can undergo these operations to fulfill the lie definition. This is useful; it is equally important to show this origin in the overall product. It was arrived at by substituting $P'Q$ for $P'$ in $P'^TK - KP' = 0$, which is a modification of what is presented above. However, if Q is not antisymmetric, this expression becomes true of upper X.

$$P'^T = C^TP^TC^{*}$$

$$P'^TK = -C^TPC$$

$$Q^TP'^TK = C^TY^TC^{*}C^TP^TC^{*}C^TC=C^TYPC$$

$$KQP' = C^TCC^{\dagger}YCC^{\dagger}PC = C^TYPC$$

$$(P'Q)^TK-KQP'=0$$

An observation used in two-gate application also applies here, from two blocks down.

$$(P'^TK+KP')Q=0$$

Expositing the work done by the paper is the focus of the lines below, however I've presented the steps in reverse order to more readily demonstrate the origin of the expression.

$$-C^TYC+C^TYC=0$$

$$Q^TK+C^TYC=0$$

$$Q^TK+KQ=0$$

Such work is extendable to two gates acting on Y, presented here for completeness.

$$C_2^{\dagger}C_1^{\dagger}YC_1C_2=C_2^{\dagger}QC_2$$

$$C_2QC^{\dagger}_2 = C_1^{\dagger}YC_1$$

$$(C_2QC_2^{\dagger})^T=-C_2QC_2^{\dagger}$$

$$(C_2^T) C_2^{*} Q^T C_2^T (C_2)=-(C_2^T)C_2QC_2^{\dagger}(C_2)$$

$$Q^TK=-KQ$$

References:

[3]. Aguilar, G.; Cichy, S.; Eisert, J.; Bittel, L. Full Classification of Pauli Lie Algebras. *arXiv:2408.00081 [quant-ph]* **2024**. DOI: [10.48550/arXiv.2408.00081](https://doi.org/10.48550/arXiv.2408.00081)

##Paulie Bounty (\$250): Matrix Basis for the Classified Algebra (\#200)

*Awarded to: WEEKEAT-LIM for Pull # 223* [Direct](https://github.com/QPauLie/PauLie/pull/223)

Paulie is centred on using commutation relationships to collapse pauli-matrix based systems into more manageable data points. Its effects are felt more at scale. However, its ability to reveal simplifications in ubiquitous pauli-based systems makes it highly useful for gauging the footprint of an NMR experiment.

The first layer, in src/paulie/classifier/classification.py, uses native classification of pauli strings and returns the basis finder as an action. Mind the \_algebras\_ versus algebra in the function names of that section. Let us briefly review what this 'primer' function sees.
```
#in paulie/classifier/classification
    Morph.get_algebra_properties(self) -> tuple[TypeAlgebra,int,int]:
        """
        Returns:
            tuple[TypeAlgebra,int,int]:
            Tuple of type of algebra, number of copies of the algebra, and the size of the algebra.
        """
        type_graph, one_legs, two_legs, long_vertices = self.get_properties()
        if type_graph == TypeGraph.NONE:
            return TypeAlgebra.U, 1, 1
        if type_graph == TypeGraph.A:
            return TypeAlgebra.SO, one_legs, long_vertices + 2
        if type_graph == TypeGraph.B1:
            return TypeAlgebra.SP, one_legs, 2**two_legs
        if type_graph == TypeGraph.B2:
            return TypeAlgebra.SO, one_legs, 2**(two_legs + 3)
        if type_graph == TypeGraph.B3:
            return TypeAlgebra.SU, one_legs, 2**(two_legs + 2)

    Morph.get_properties(self) -> tuple[TypeGraph,int,int,int]:
        one_legs, two_legs, long_vertices = self.counts()
        if two_legs == 0:
            return  TypeGraph.A, one_legs, two_legs, long_vertices
        if long_vertices == 0:
            return  TypeGraph.B1, one_legs, two_legs, long_vertices
        if long_vertices == 3:
            return  TypeGraph.B3, one_legs, two_legs, long_vertices
        if long_vertices == 4:
            return  TypeGraph.B2, one_legs, two_legs, long_vertices

    Morph.counts(self) -> tuple[int,int,int]:
    for i, leg in enumerate(self.legs):
            if i == 0:
                continue
            if len(leg) == 1:
                one_legs += 1
            if len(leg) == 2:
                two_legs += 1
            if len(leg) > 2:
                long_vertices += len(leg)
    if long_vertices == 0 and two_legs == 1:
            two_legs = 0
            long_vertices = 2
        if long_vertices > 0 and two_legs == 0:
            long_vertices += 1
        if long_vertices == 0 and two_legs == 0 and one_legs == 1:
            long_vertices = 1
        if long_vertices == 0 and two_legs == 0 and one_legs == 2:
            long_vertices = 1
            one_legs = 2
        return one_legs, two_legs, long_vertices

class Morph:
    """
    Stores the structural data of a canonical graph.
    """
    def __init__(self, legs:list[list[PauliString]], independents:list[PauliString],
        generators:list[PauliString]) -> None:
        """
        Initialize the structural data of the graph.

        Args:
           legs (list[list[PauliString]]): The center of the graph followed by the list of legs.
           independents (list[PauliString]): List of independent vertices.
           generators (list[PauliString]): list of generators
        """
        self.legs = legs # center is zero leg
        self.independents = set(independents)
        self.generators = generators
```
The most important number to grab is the scripted letter of the previous background section and only the one-length and multi-length, but not two-length, chains. We can immediately recognize the size of the algebra is the dimension numbers of the respective groups as multi-length chains, and one-length chains represent control bits. In fact, those control bits are explicitly called copies of the algebra.

This requires the program looking through lists of lists for the legs, counting up the non-duplicates in each via set(independents) and tracking the strings creating the set. The length of the inner lists is the length of the legs. Get properties is only a summary for the symmetry group that becomes the scripted letter at the end. counts() has some basic rules like turning two control bits into a seperation of at least two other qubits, or else they could contract. Another basic rule is that giving it one pauli matrix to work with causes it to assume one leg of >2 is hiding, which is very insightful, $\{Z_0$,$Z_0X_1$,$X_0Z_1\}$ is an example solution that would seemingly have single bit generator.
```
#src/paulie/classifier/classification
Morph.get_algebra_basis(self) -> np.ndarray:
            type_alg, nc, size = morph.get_algebra_properties()
            multiplier = nc if nc == 1 else 2 ** (nc - 1)
            key = (type_alg, size)
            algebra_map[key] = algebra_map.get(key, 0) + multiplier
            [...]
            for (type_alg, size), count in algebra_map.items():
                multipliers.append(count)
                groups.append(type_alg)
                sizes.append(size)
            return get_algebras_basis(multipliers, groups, sizes)
```
A dictionary key of ordered pair, (scripted letter, matrix dimension), is created. It also reads how many control bits exist for how many times the operation needs to be repeated as a direct sum. It reads the dictionary back for dimensions, matrix group - the key, and also the number of direct sums that key stored. For interested parties, .get(key) + multiplier is the recursive step that stores how many direct sums have to happen at each level, and multipliers, groups, sizes are lists for subgraphs since I hid from the code block an iteration over morphologies.

get_algebras_basis calls two functions, get_n_basis and get_group_basis, also in src/paulie/common/algebra_basis.py. get_n_basis is a control flow branch matching every lie group's letter to its integer matrix dimension. That same dimension is also the number of generators creating the group, because it is generated element-wise. There are three such functions. Get basis really is the promised matrix build, element by element, calling get_n_basis itself for the size of each matrix.
```
#in src/paulie/common/algebra_basis.py
def get_n_so_basis(n):
    return n * (n - 1) // 2
def get_n_su_basis(n):
    return n**2 - 1
def get_n_sp_basis(n):
    return n * (2 * n + 1)
```
The purpose of get_n_basis in get_algebras_basis is to translate over every generated element to where it belongs in a block-dimensional basis. Given two sp(n) branches spanning off the same matrix, the first takes up n\*\*2-1 rows of length \n\*\*2-1\ then at coordinate (n\*\*2-1),(n\*\*2-1) is placed the elements up to 2(n\*\*2-1),2(n\*\*2-1). Its purpose is to calculate indexes to copy the solution into.

To illustrate my point about the straightforwardness of basis generation, the following code uses an upper triangular =1 and lower triangular =-1 for the $\mathfrak{so}$ group. It creates identical matrices this way $\lfloor n*(n-1)/2 \rfloor$ times.
```
#in src/paulie/common/algebra_basis
def get_so_basis(n: int) -> np.ndarray:
    dim: int = get_n_so_basis(n)
    basis = np.zeros((dim, n, n), dtype=np.complex128)
    rows, cols = np.triu_indices(n, k=1)
    k = np.arange(dim)
    basis[k, rows, cols] = 1.0
    basis[k, cols, rows] = -1.0
    return basis
```
The requirements for the SU basis is a traceless matrix, whose adjoint is the matrix itself and it must be traceless. This permits an upper triangular -j and lower triangular +j, that change signs then transpose to equal the same matrix.
$$\mathfrak{su}(N)≡\{P∈C|P^{\dagger}=P,Tr(P)=0\}$$
This is implemented as an "imaginary tails" matrix that can modify all off-diagonal elements to be -j. It does this in a sequence of redundant rewrites down triangular indeces but repeated in diagonal indexing instead of creating imaginary and real blocks. These parts of the submission are not worth examining. The purpose of presenting the hermitian property preservation of the group is because the exponentiation of a hermitian matrix is a unitary, so presenting the $U^{\dagger}U=1$ property does not make sense.

Finally, to conclude, here is how get_n_.._basis was used in assembling these matrices into a product. It adds the size of the basis to n_basis_per_group, with total_dim and total_n allocating memory, so it can add to gen_offset the block generated. diag_offset corresponds to adding each block's coordinates down the diagonal of the large basis matrix.
```
def get_algebras_basis(
        multipliers: list[int],
        groups: list['TypeAlgebra'],
        sizes: list[int]
    ) -> np.ndarray:
    block_sizes = []
    n_basis_per_group = []
    total_dim = 0
    total_n = 0

    for i in range(n_pairs):
        actual_block_size = sizes[i] * 2 if groups[i] == TypeAlgebra.SP else sizes[i]
        block_sizes.append(actual_block_size)

        n_basis = get_n_basis(groups[i], sizes[i])
        n_basis_per_group.append(n_basis)

        total_dim += n_basis * multipliers[i]
        total_n += actual_block_size * multipliers[i]
    basis = np.zeros((total_dim, total_n, total_n), dtype=np.complex128)
    en_offset = 0  # Tracks depth in the 3D tensor
    diag_offset = 0  # Tracks slide down the 2D diagonal
    for i in range(n_pairs):
        base_matrices = get_group_basis(groups[i], sizes[i])
        # Stamp this basis out for however many copies (multipliers) we need
        for _ in range(multipliers[i]):
            n_gen = n_basis_per_group[i]
            b_size = block_sizes[i]
            # Vectorized assignment: Drop the entire chunk of matrices into the correct slice
            basis[
                gen_offset : gen_offset + n_gen,
                diag_offset : diag_offset + b_size,
                diag_offset : diag_offset + b_size
            ] = base_matrices

            gen_offset += n_gen
            diag_offset += b_size

    return basis
```

#Gaps and Personal Development

##Statement on Applicability🏆🏮

Such work following closely in the Paulis is demonstration of the ability to establish NMR and EPR basis sets, at-will. Well-known rules can be explained with interrogation, and further work in novel signals can be constructed at-sight from the critical step without reconstructing prior mixing steps. There are also small skills like opening dev environments in virtual python platforms (no special permissions) and opening files from a quantum computing experiment in Python to check for their variables. These are all going to resemble typical laboratory equipment backends, since they're constructed from similar components.

##Julia 💫
Programming language Julia was well represented at UnitaryHack this year, with bold offerings like Piccolo.jl solving for control pulses that implement quantum operations, PauliStrings.jl being a classic take in a new language on reducing the pauli matrices into well-known decompositions. The .jl advantage is to compile code as it runs, while Python has a machine compilation step to turn it into computer-readable code. However, what peaked my interest in Julia was the question of why so many languages were using C++ backends in Python libraries around quantum computing. This is what was returned in terms of speed, but also the ability to assign cores faster this way. Julia seems like the best next-step for any scientific computing division, because any C++ specialized development will usually be rolled out company-wide with validation steps and a dense pipeline. Julia can take advantage of stronger computing techniques and as a lightweight language with pythonic syntax in case of a quick, bench-scale prototype. This is particularly good for microfluidics applications, which are blurring the lines in computation and on-site testing, which has a huge margins problem in terms of how much computing a lab-on-a-chip would actually need to do.

The Control-flow types are very cool. Short-circuiting OR, `||`, will evaluate the condition to its left and if it's False, will evaluate the condition on the right. If the left is true, OR is already true so this saves computation. On a bigger scale, it catches edge cases where several lines of code can pass through a check but have conditional discards in case of data that is actually interfering with an experiment. Short-circuiting AND, `&&`, evaluates the left piece of code for truth and doesn't proceed if it sees False. This is most directly applicable as positive selection for a result.

The Julia functions are the strongest, most versatile argument for transitioning. You can design custom data types with `mutable struct [...] end` and a function is written with different bodies according to the data type it recieves. For the latter point, `Gather_integrals(Int64)` can sum up all the calculations you did into energy levels but `Gather_integrals(String)` can just have a body, `Return "this is not a valid data type"`. Of course, on a more sophisticated level, the mutable struct Tensor, and the mutable struct Operator are respectively a list of of three numbers and a list of eigenvalues placed into a matrix. Thus, one only needs to write orientation randomization into the latter to recover the former in a function at the end, while both have similar interactions with field strength.

##Pandas 🎋🐼
A python library used data sciences because of its ability to work with large amounts of numbers, this is an innevitable skill for extracting data from laboratory instruments. This is a brief tutorial on the most used functions.

Uses sanity checks from https://ez.restek.com/proezlc optimized methods, which you may freely use to replicate the results of these simulations in your own script!



In [ ]:
column = [[50,3.0,2.7],"Raptor Biphenyl"]

time = np.linspace(73,780,707,dtype=int)
flow_rate = 0.8 #in mL/min
flow_rate /= 60 #mL/s
elution_volume = time*flow_rate

series = pd.Series(elution_volume,index=time)
if series.iloc[60] == series.loc[133]:
    #one minute of elution time as a volume
    print(series.loc[133] - series.loc[73])
    #desired to check for an exact t_r
    series.loc[73] = ((0.21 + 1.02)*60)*flow_rate
    print(series[series < 1.00])

expected_eluotropic_strengths = {"Standards": [
"SuperTox","Impurity_1","Impurity_2","WeedZ"
],
"reported t_r'":np.array([61, 300, 450, 635
])
}
outcomes = {"Eluted":[
"compound_1","compound_2","compound_3","compound_4"
],
"Time_Eluted": np.array([
61, 300, 450, 635
]) + 12
}
outcomes_df = pd.DataFrame(outcomes, index=outcomes["Time_Eluted"])
expected_eluotropic_strengths["reported t_r'"] += 12
expecteds_df = pd.DataFrame(expected_eluotropic_strengths,
               index=expected_eluotropic_strengths["reported t_r'"]
)
expecteds_df["Standards"] = expecteds_df["Standards"].str.lower()
ramp_timestep_1percent = 8
gradient_translation = [val/ramp_timestep_1percent
                        for val in outcomes["Time_Eluted"]
]
outcomes_df["grad_equiv"] = gradient_translation
if outcomes_df["Time_Eluted"].all() == expecteds_df["reported t_r'"].all():
    outcomes_df["Eluted"] = expecteds_df["Standards"]
    print(outcomes_df)
if ((outcomes_df["Time_Eluted"].sum()- outcomes_df["Time_Eluted"].max())
     < outcomes_df["Time_Eluted"].max()):
    print("Please optimize the run")
if (outcomes_df["Time_Eluted"].mean() == outcomes_df["Time_Eluted"].min() or
   expecteds_df["Standards"].count() != outcomes_df["Eluted"].count()):
    print("Please use weaker gradients")
print("Your mean experiment costs")
print(outcomes_df.mean(numeric_only = True))

master_expt = pd.DataFrame(elution_volume,index=time)
time_df = pd.DataFrame(np.full(shape=[707],fill_value=None), index = time)
for i in outcomes_df["Time_Eluted"]:
    time_df.loc[i] = outcomes_df.loc[i,"Eluted"]
master_expt["Compound Identity"] = time_df
master_expt = master_expt.fillna({"Compound Identity":"blank"})
time_df = (
pd.DataFrame(np.linspace(73,780,707,dtype=float) / ramp_timestep_1percent,
             index = time)
)
master_expt["grad_equiv"] = time_df
master_expt["volume_eluted"] = master_expt.iloc[0:708,0]
master_expt = master_expt.drop(columns = 0)
for i in range(72,11,-1):
    blank = {"Compound Identity": "blank",
    "grad_equiv": i / ramp_timestep_1percent,
    "volume_eluted": i*flow_rate
    }
    time_df = pd.DataFrame(blank,index = [i])
    master_expt= pd.concat([time_df,master_expt])
master_expt = master_expt.dropna(subset=["Compound Identity"])
master_expt["Compound Identity"] = master_expt["Compound Identity"].astype(str)
master_expt["Compound Identity"] = master_expt["Compound Identity"].replace(
                                    {"blank":"eluent"}
)

detection_groups = master_expt.groupby("Compound Identity")
print(detection_groups["volume_eluted"].mean())
exclude_unretained = master_expt[(master_expt["Compound Identity"] != "eluent")|
                               (master_expt["volume_eluted"] > 12*flow_rate)
]
analytes = master_expt[(master_expt["Compound Identity"] != "eluent") &
                       (master_expt["Compound Identity"] != "impurity_1") &
                       (master_expt["Compound Identity"] != "impurity_2")
]
print(analytes[["Compound Identity", "volume_eluted", "grad_equiv"]])
print(analytes.iloc[0:2:1,0:2])
master_expt = master_expt.to_csv("data.csv")
master_expt = pd.read_csv("data.csv",index_col=0)
print(master_expt.loc[60:73].to_string)

##SymPy🀄

Advertised alongside a paid Mathematica subscription in Jones, Hore, & Wimperis' "NMR: The Toolkit- How Pulse Sequences Work" for eigenvector finding, more in-depth personal research has revealled it to be block-translations of common differential equation outcomes. It takes solutions like exponential curves and first-order sine/cosine solutions, and turns parts of each equation into these. Working this way would have obvious weaknesses with discrete sum methods. Following work with teacups and Unitary Hack's eigenvector Numpy implementations, I believe I might be in a proper position to develop or borrow solutions from common interests in bridging this gap.

##References

[1]. Williams, C.P. Quantum Information. *Explorations in quantum computing*, 2nd ed.; Springer London, 2011; pp 403-482. DOI: [https://doi.org/10.1007/978-1-84628-887-6](https://doi.org/10.1007/978-1-84628-887-6)

[2]. Manna, S.; Das Bhowmik, A. Single-shot Antidistinguishability of Unitary Operations. *Physical Review A* **2026**, 113(2), 022218. DOI:https://doi.org/10.1103/d183-k1x3 ArXiv:2510.14609

[3]. Aguilar, G.; Cichy, S.; Eisert, J.; Bittel, L. Full Classification of Pauli Lie Algebras. *arXiv:2408.00081 [quant-ph]* **2024**. DOI: [10.48550/arXiv.2408.00081](https://doi.org/10.48550/arXiv.2408.00081)